# 03. Ramsey 탐색기와 독립 인증 검증기

## 목표
완전 그래프 edge의 2-coloring에서 단색 삼각형이 없는 certificate를 찾고 별도 함수로 검증합니다. 작은 고전 결과 `R(3,3)=6`을 통해 증명 탐색과 proof checking의 차이를 배웁니다. 논문의 다색 초지수 하한 재현은 아닙니다.

In [ ]:
from itertools import combinations

def edges(vertex_count):
    return list(combinations(range(vertex_count), 2))

def has_monochromatic_triangle(vertex_count, coloring):
    for a, b, c in combinations(range(vertex_count), 3):
        colors = {coloring[(a, b)], coloring[(a, c)], coloring[(b, c)]}
        if len(colors) == 1:
            return True
    return False

def verify_certificate(vertex_count, coloring):
    expected = set(edges(vertex_count))
    return set(coloring) == expected and set(coloring.values()) <= {0, 1} and not has_monochromatic_triangle(vertex_count, coloring)

In [ ]:
def find_triangle_free_coloring(vertex_count):
    edge_list = edges(vertex_count)
    coloring = {}

    def creates_triangle(edge, color):
        left, right = edge
        for middle in range(vertex_count):
            if middle in edge:
                continue
            first = tuple(sorted((left, middle)))
            second = tuple(sorted((right, middle)))
            if coloring.get(first) == color and coloring.get(second) == color:
                return True
        return False

    def search(index):
        if index == len(edge_list):
            return dict(coloring)
        edge = edge_list[index]
        for color in (0, 1):
            if not creates_triangle(edge, color):
                coloring[edge] = color
                result = search(index + 1)
                if result is not None:
                    return result
                del coloring[edge]
        return None

    return search(0)

for n in (5, 6):
    certificate = find_triangle_free_coloring(n)
    print(f'K_{n}: found={certificate is not None}', end='')
    if certificate is not None:
        print(f', verified={verify_certificate(n, certificate)}')
    else:
        print()

In [ ]:
certificate_k5 = find_triangle_free_coloring(5)
print(certificate_k5)

tampered = dict(certificate_k5)
tampered.pop(next(iter(tampered)))
print('누락된 edge가 있는 certificate:', verify_certificate(5, tampered))
# 탐색기는 복잡해도 verifier는 작고 명확하게 유지할 수 있습니다.
# Lean kernel과 형식 certificate의 관계도 이 분리를 훨씬 엄밀하게 확장합니다.

## 한계와 확장

- K6에서 해를 못 찾았다는 실행 결과만으로 일반적인 수학 증명이 완성되는 것은 아닙니다. 탐색 공간의 완전성과 구현 정확성을 증명하거나 trusted checker로 옮겨야 합니다.
- `k`색으로 확장하고 작은 `R_k(3)` 하한 certificate를 찾아 보세요.
- 탐색기와 verifier를 다른 구현으로 작성하면 공통 bug 위험을 줄일 수 있습니다.
- Lean 형식화에서는 자연어 명제가 정확히 옮겨졌는지도 별도로 검토해야 합니다.